# Lab 4: Scanning a RAG system for vulnerabilities

This lab runs as an instructor demo. You do not need to run the scan yourself, and the
install cell below deliberately does not install Giskard. Here is why, because it is a
better lesson than the scan itself.

Giskard's classic API (`giskard.Model`, `giskard.Dataset`, `giskard.scan`) belongs to its
2.x line. Every 2.x release declares `Requires-Python <3.13`. Colab now runs Python
3.13, so `pip install giskard` resolves to **3.0.0**, a ground-up rewrite with a different
API, and pinning `giskard==2.19.2` fails with "no matching distribution" — which reads
exactly like the package was pulled, though it is still on PyPI and installs fine on 3.12.

That is worth sitting with. Nothing about the model, the corpus or the code changed. A
runtime upgrade underneath it took the tool away, and the error message pointed at the
wrong cause. Pin your evaluation stack, and pin the Python it runs on.

So: the system under test gets built here, live, exactly as in Lab 3. The scan was run
ahead of time on Python 3.12 and its report is downloaded below.


In [ ]:
%pip install -q \
  litellm==1.102.0 "openai>=2.20.0,<3.0.0" \
  llama-index==0.14.24 llama-index-readers-file fonttools \
  llama-index-vector-stores-chroma==0.6.0 \
  llama-index-embeddings-litellm==0.6.0 llama-index-llms-openrouter \
  llama-index-retrievers-bm25 bm25s PyStemmer \
  chromadb==1.5.9 pandas==2.2.3 pyarrow ipython-autotime

%load_ext autotime


In [ ]:
import os
import urllib.request

import pandas as pd
import Stemmer
import chromadb

from llama_index.core import VectorStoreIndex, Settings
from llama_index.core.schema import TextNode
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.litellm import LiteLLMEmbedding
from llama_index.llms.openrouter import OpenRouter
from IPython.display import HTML, display


In [ ]:
# Workshop API key. Paste the key handed out in the room; do not commit it.
# Phase 8 replaces this cell with a call to the hub's /api/env endpoint.
os.environ["OPENROUTER_API_KEY"] = ""

# Fail here, clearly, rather than 20 cells later. With an empty key, llama-index falls
# back to its OpenAI client and reports "Missing credentials ... set OPENAI_API_KEY",
# which sends you looking for the wrong key entirely.
assert os.environ.get("OPENROUTER_API_KEY"), (
    "Paste the workshop key into the line above before running the rest of the notebook.")


GPT_MODEL = 'openai/gpt-4.1-mini'
CONTEXT_WINDOW = 128_000
MAX_TOKENS = 1024

# context_window and max_tokens are set explicitly, and both matter.
# llama-index's OpenRouter class defaults to context_window=3900 and max_tokens=256.
# Five retrieved chunks are about 4,200 tokens, so with the default window llama-index
# silently splits the context and answers over several "refine" passes: slower, and a
# worse answer, with no warning. Setting a real window makes it one call.
# The generator is also deliberately NOT a reasoning model. gpt-5-mini spends the 256-token
# default budget on reasoning and returns llama-index's "Empty Response", which scores zero
# on faithfulness and answer relevancy while looking like a retrieval problem.

EMBED_MODEL = 'openrouter/openai/text-embedding-3-small'
CORPUS_URL = "https://github.com/ndecavel/tdwi-workshop-labs/releases/download/corpus-2026-09-20/improved_nodes.parquet"
REPORT_URL = "https://github.com/ndecavel/tdwi-workshop-labs/releases/download/corpus-2026-09-20/scan_report.html"

llm = OpenRouter(model=GPT_MODEL, api_key=os.environ["OPENROUTER_API_KEY"],
                 context_window=CONTEXT_WINDOW, max_tokens=MAX_TOKENS)
Settings.llm = llm
Settings.embed_model = LiteLLMEmbedding(model_name=EMBED_MODEL, embed_batch_size=100)


## 1. Build the system under test

The same corpus and the same hybrid retriever as Lab 3. A scan is only meaningful against
the thing you actually ship, so this is not a simplified stand-in.


In [ ]:
urllib.request.urlretrieve(CORPUS_URL, "/content/improved_nodes.parquet")
corpus = pd.read_parquet("/content/improved_nodes.parquet")

nodes = [
    TextNode(id_=row.id, text=row.text, embedding=list(row.embedding),
             metadata={"file_name": row.file_name, "page_label": row.page_label})
    for row in corpus.itertuples()
]

chroma_client = chromadb.EphemeralClient()
chroma_collection = chroma_client.get_or_create_collection(
    name="giskard_corpus", configuration={"hnsw": {"space": "cosine"}})
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
BATCH = 4000
for i in range(0, len(nodes), BATCH):
    vector_store.add(nodes[i:i + BATCH])

index = VectorStoreIndex.from_vector_store(vector_store, embed_model=Settings.embed_model)

hybrid_retriever = QueryFusionRetriever(
    [index.as_retriever(similarity_top_k=5),
     BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=5,
                                 stemmer=Stemmer.Stemmer("english"), language="english")],
    mode="reciprocal_rerank", num_queries=1, similarity_top_k=5, use_async=True,
)
hybrid_query_engine = RetrieverQueryEngine.from_args(hybrid_retriever, llm=llm)

print(f"{chroma_collection.count()} chunks indexed, hybrid engine ready")


In [ ]:
!wget -q https://raw.githubusercontent.com/ndecavel/tdwi-workshop-labs/main/data/behr/Golden_Test_Data_DeepEval.csv

golden_df = pd.read_csv('/content/Golden_Test_Data_DeepEval.csv')
golden_df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'], inplace=True)
golden_df = golden_df.rename(columns={'input':'question', 'expected_output':'ground_truth'})

examples = golden_df.sample(5, random_state=42)["question"].tolist()
for q in examples:
    print("Q:", q)
    print("A:", str(hybrid_query_engine.query(q))[:300], end="\n\n")


## 2. What the scan does

Giskard probes a model the way an adversary would, rather than scoring it against a fixed
answer key. For a RAG system the interesting detector is **hallucination**: it generates
questions designed to pull the model past the edge of its corpus, then checks whether the
model invents an answer or admits it does not know.

This is the code that produced the report below. It is shown, not run, for the Python
version reason at the top of this notebook.

```python
import giskard
from giskard.llm import set_llm_model, set_embedding_model

# Giskard delegates to LiteLLM, so its own judge and embedding calls go through
# OpenRouter too. Its defaults are gpt-4o and text-embedding-3-small on OpenAI direct,
# which is the only reason this lab ever needed an OpenAI key.
set_llm_model("openrouter/openai/gpt-4.1-mini")
set_embedding_model("openrouter/openai/text-embedding-3-small")

def model_predict(df: pd.DataFrame):
    """Takes a DataFrame of inputs, returns one output per row."""
    return [str(hybrid_query_engine.query(q)) for q in df["question"]]

giskard_model = giskard.Model(
    model=model_predict,
    model_type="text_generation",
    name="BEHR Paint Technical Data Sheet Question Answering",
    description="Answers questions about BEHR paint technical data sheets.",
    feature_names=["question"],
)

giskard_dataset = giskard.Dataset(pd.DataFrame({"question": examples}), target=None)

report = giskard.scan(giskard_model, giskard_dataset, only="hallucination")
report.to_html("scan_report.html")
```

If a scan ever fails to parse its own results, the cause is usually structured output:
`set_llm_model(..., disable_structured_output=True)`, or pick a model whose OpenRouter
listing shows native strict support.


In [ ]:
urllib.request.urlretrieve(REPORT_URL, "/content/scan_report.html")

with open("/content/scan_report.html") as fh:
    display(HTML(fh.read()))


## 3. Reading the report

The scan ran two hallucination detectors over five golden questions: about 30 calls to the
model and 22 judge calls, two minutes in total. It came back with **one major issue**.

**Sycophancy, 2 failing samples.** The detector asks the same question twice, biased in
opposite directions, and compares the answers. Ours contradicted itself. One of the failing
pairs is about BEHR Marquee dry-to-touch time: ask it leadingly one way and it agrees, ask
it leadingly the other way and it agrees with that too.

That is worth more than a passing score would be. Three things to take from it:

1. **The golden set never caught this.** Every question in `Golden_Test_Data_DeepEval.csv`
   is asked neutrally, once. The failure only appears when the same fact is approached from
   two directions, which is how a real user with a hunch asks.
2. **Retrieval was not the problem.** The hybrid retriever found the right data sheet both
   times. The model then bent the answer to match the question's framing. More retrieval
   work would not have fixed it.
3. **The fix is upstream of the retriever.** A system prompt that explicitly permits
   disagreeing with the user, and requires the answer to come from the retrieved text,
   addresses this class of failure. Then re-run the scan and see whether it holds.

`report.generate_test_suite()` turns this into a reusable suite, which is the real payoff: a
scan you run once is a report, and a scan you run on every change is a regression test.
